# [실습] Skill 패턴 (Progressive Disclosure)

에이전트의 Skill은 필요한 지침만 단계적으로 로드해 컨텍스트 사용량을 줄이는 패턴입니다.

Anthropic의 [Agent Skills 표준](https://agentskills.io)에 기반한 `SKILL.md` 형식으로 `skills/` 폴더에 도메인 지식을 정의합니다.
에이전트는 필요할 때 해당 스킬을 로드하여 작업을 수행합니다.

## 학습 목표

- Skill 패턴이 도구/지식 증가에 따른 컨텍스트 낭비를 줄이는 방식 확인
- `SKILL.md`의 YAML frontmatter와 Markdown body 구조 구분
- `scan_skills`, `load_skill`, `run_command`로 이어지는 스킬 로드 파이프라인 구현
- `SkillMiddleware`로 시스템 프롬프트에 스킬 목록 주입, `tools`로 도구 등록
- 3단계 progressive disclosure 흐름 확인: 메타데이터, 본문, 참고 파일

## 환경 설정

In [1]:
%pip install langchain langchain-openai langgraph pyyaml -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv('.env', override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 설정 완료!')

OpenAI API 키 설정 완료!


In [3]:
from select_model import load_model

# 실습용 모델
open_model = load_model(platform='vllm', enable_thinking=True, max_tokens=16384, temperature=1.0, top_p=0.95, presence_penalty=1.5, extra_body={"top_k": 20, "min_p": 0.0, "repetition_penalty": 1.0})
print('open_model:', type(open_model).__name__)

open_model: ChatOpenAIWithReasoning


---
## 1. Skill 패턴의 필요성

에이전트에 도메인 지식(DB 스키마, 비즈니스 로직, 가이드라인 등)을 제공하는 방식은 다음과 같습니다.

| 방식 | 장점 | 단점 |
|------|------|------|
| 시스템 프롬프트에 전부 포함 | 항상 접근 가능 | 컨텍스트 낭비, 도메인 추가 시 선형 증가 |
| Skill 패턴 (Progressive Disclosure) | 필요할 때만 로드 | 첫 호출 시 스킬 로드 단계 추가 |

```
전체 포함 방식: 모든 도메인 지식을 시스템 프롬프트에 포함하면 매번 전체 토큰을 컨텍스트에 저장
Skill: 메타데이터 목록만 주입하고 필요한 스킬만 로드하므로 컨텍스트 절약
```

### Agent Skills 표준 (agentskills.io)

이 실습에서는 Agent Skills의 `SKILL.md` 형식으로 스킬을 연결합니다.

스킬은 `SKILL.md` 파일로 정의하며, YAML frontmatter(메타데이터)와 Markdown body(지침)로 구성됩니다.

```yaml
---
name: skill-name              # 스킬 식별자 (소문자, 하이픈)
description: 스킬 설명         # Agent가 자동 호출 판단에 사용
metadata:
  type: cli-tool               # cli-tool | prompt-bundle | file-template
---

Markdown 본문: DB 스키마, 비즈니스 로직, 스크립트 사용법 등
```

### Skill 유형 예시

스킬 유형은 실행 방식에 따라 다음처럼 구분합니다.

| 유형 | 구성 내용 | 실행 방식 |
|------|-------------|----------|
| CLI Tool | CLI 명령어 사용법 + 스크립트 | `run_command`로 실행 |
| Prompt Bundle | LLM 지침/체크리스트 | LLM이 직접 수행 |
| File Template | 파일 경로 + 양식 설명 | 파일 읽고 LLM이 채움 |

---
## 2. 스킬 폴더 구조

```
skills/
- sales_analytics/           (CLI Tool: 매출 DB 스키마 + 쿼리 스크립트)
    - SKILL.md               (YAML frontmatter + 테이블 구조, 비즈니스 로직)
    - setup.py               (SQLite 샘플 데이터 생성)
  - query.py               (SQL 쿼리 실행 유틸리티)
- inventory_management/      (CLI Tool: 재고 DB 스키마 + 쿼리 스크립트)
    - SKILL.md
    - setup.py
  - query.py
- customer_report/           (Prompt Bundle + File Template)
    - SKILL.md               (보고서 작성 가이드라인, 스타일 규칙)
    - generate.py            (매출/재고 DB 연계 보고서 생성)
```

각 폴더의 `SKILL.md`는 Agent Skills 표준을 따릅니다:

```yaml
---
name: sales-analytics                    # 스킬 식별자
description: 매출 데이터 분석을 위한 ...    # 모델의 스킬 선택에 사용하는 설명
metadata:
  type: cli-tool
---

## Tables                                # Markdown 본문: 상세 도메인 지식
### customers
| 컬럼 | 타입 | 설명 |
...
```

### 2-1. 샘플 DB 생성

스킬 스크립트를 테스트하기 위해 먼저 샘플 데이터베이스를 생성합니다.

In [ ]:
# python skills/sales_analytics/setup.py
# python skills/inventory_management/setup.py

In [ ]:
# 쿼리 테스트
# python skills/sales_analytics/query.py "SELECT * FROM customers LIMIT 3"

---
## 3. Skill 시스템 구현

### 구성 요소

1. `scan_skills()`: `skills/` 폴더의 `SKILL.md` YAML frontmatter를 파싱하여 스킬 레지스트리 생성
2. `load_skill` 도구: 스킬의 Markdown body와 실행 가능 파일 목록 반환
3. `run_command` 도구: 스킬 스크립트 실행 (`.venv` Python 자동 사용)
4. `SkillMiddleware`: `wrap_model_call`로 시스템 프롬프트에 스킬 목록 주입, `tools = [load_skill]`로 도구 자동 등록

In [4]:
import yaml
from pathlib import Path

SKILLS_DIR = Path("skills")

def scan_skills(skills_dir: Path) -> dict[str, dict]:
    """SKILL.md의 frontmatter와 본문으로 스킬 레지스트리를 생성합니다."""
    skills = {}
    if not skills_dir.exists():
        return skills
    for d in sorted(skills_dir.iterdir()):
        if not d.is_dir():
            continue
        skill_file = d / "SKILL.md"
        if not skill_file.exists():
            continue
        content = skill_file.read_text(encoding="utf-8")
        # --- frontmatter --- 파싱
        if not content.startswith("---"):
            continue
        parts = content.split("---", 2)
        if len(parts) < 3:
            continue
        meta = yaml.safe_load(parts[1])
        body = parts[2].strip()
        name = meta.get("name", d.name)
        skills[name] = {
            "name": name,
            "description": meta.get("description", ""),
            "path": str(d),
            "folder": d.name,
            "meta": meta,
            "body": body,
        }
    return skills


SKILLS = scan_skills(SKILLS_DIR)
print(f"스캔된 스킬 {len(SKILLS)}개:")
for name, s in SKILLS.items():
    skill_type = s["meta"].get("metadata", {}).get("type", "unknown")
    print(f"  - {name} [{skill_type}]: {s['description'][:60]}...")

스캔된 스킬 5개:
  - customer-report [prompt-bundle]: 고객 분석 보고서 생성 가이드라인과 마크다운 템플릿 (매출 DB + 재고 DB 연계 분석)...
  - data-viz [prompt-bundle]: 수치 데이터를 matplotlib 파이썬 코드로 차트화한다. 막대/선/원형 등 실제 데이터 시각화가 필요할 ...
  - image-gen [prompt-bundle]: 데이터 차트가 아닌 컨셉/일러스트/표지 이미지를 생성형 모델로 만든다. 리포트 표지, 삽화, 분위기 이미지가...
  - inventory-management [cli-tool]: 재고 관리를 위한 DB 스키마와 SQL 쿼리 스크립트 (products, warehouses, invento...
  - sales-analytics [cli-tool]: 매출 데이터 분석을 위한 DB 스키마와 SQL 쿼리 스크립트 (customers, orders, order_...


In [5]:
import subprocess
import re
import os
import sys
import platform
from langchain.tools import tool

IS_WINDOWS = platform.system() == "Windows"
WORKSPACE = Path.cwd().resolve()


def _safe_path(p: str) -> Path:
    """WORKSPACE 내부 경로로 정규화합니다."""
    target = (WORKSPACE / p).resolve()
    if not str(target).startswith(str(WORKSPACE)):
        raise ValueError(f"작업 디렉터리 밖 경로 접근 불가: {p}")
    return target


@tool
def load_skill(skill_name: str) -> str:
    """스킬의 상세 내용(SKILL.md body)을 로드합니다. 해당 도메인의 작업을 수행하기 전에 호출하세요.

    Args:
        skill_name: 로드할 스킬 이름 (YAML frontmatter의 name 필드)
    """
    if skill_name not in SKILLS:
        available = ", ".join(SKILLS.keys())
        return f"스킬 '{skill_name}'을 찾을 수 없습니다. 사용 가능: {available}"

    skill = SKILLS[skill_name]
    body = skill["body"]

    # 실행 가능한 스크립트 목록 첨부
    skill_dir = Path(skill["path"])
    scripts = [f.name for f in skill_dir.iterdir() if f.is_file() and f.suffix == ".py"]
    if scripts:
        body += "\n\n## 실행 가능한 스크립트\n"
        py_hint = ".venv\\Scripts\\python" if IS_WINDOWS else ".venv/bin/python"
        for s in scripts:
            body += f"- `{skill['folder']}/{s}`: `{py_hint} skills/{skill['folder']}/{s}`\n"

    # 참고 파일 목록 첨부
    ref_files = [
        f"skills/{skill['folder']}/{f.relative_to(skill_dir)}".replace("\\", "/")
        for f in sorted(skill_dir.rglob("*"))
        if f.is_file() and f.name != "SKILL.md" and f.suffix != ".py"
    ]
    if ref_files:
        body += "\n\n## 참고 파일\n필요할 때 read_file로 읽으세요. 작업에 필요한 파일만 로드합니다.\n"
        for rf in ref_files:
            body += f"- `{rf}`\n"

    return body


@tool
def write_file(path: str, content: str) -> str:
    """파일을 작성(덮어쓰기)합니다. 부모 디렉터리는 자동 생성됩니다.
    긴 내용의 SKILL.md, 스크립트 등을 만들 때 셸을 거치지 말고 이 도구를 사용하세요.

    Args:
        path: 작업 디렉터리 기준 상대 경로 (예: "skills/my_skill/SKILL.md")
        content: 파일에 쓸 전체 내용
    """
    try:
        target = _safe_path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return f"완료: {target.relative_to(WORKSPACE)} 작성 완료 ({len(content)}자)"
    except Exception as e:
        return f"[ERROR] {e}"


@tool
def read_file(path: str) -> str:
    """파일 내용을 읽어 반환합니다.

    Args:
        path: 작업 디렉터리 기준 상대 경로
    """
    try:
        return _safe_path(path).read_text(encoding="utf-8")
    except Exception as e:
        return f"[ERROR] {e}"


@tool
def list_files(path: str = ".") -> str:
    """디렉터리의 파일/하위 폴더 목록을 반환합니다.

    Args:
        path: 작업 디렉터리 기준 상대 경로 (기본값: 현재 디렉터리)
    """
    try:
        target = _safe_path(path)
        if not target.is_dir():
            return f"[ERROR] 디렉터리가 아님: {path}"
        entries = sorted(p.name + ("/" if p.is_dir() else "") for p in target.iterdir())
        return "\n".join(entries) if entries else "(empty)"
    except Exception as e:
        return f"[ERROR] {e}"


# OS에 맞는 venv Python 경로 탐색, 없으면 현재 인터프리터 사용
if IS_WINDOWS:
    _candidate = os.path.join(os.getcwd(), ".venv", "Scripts", "python.exe")
else:
    _candidate = os.path.join(os.getcwd(), ".venv", "bin", "python")
_VENV_PYTHON = _candidate if os.path.isfile(_candidate) else sys.executable

# 윈도우(.venv/Scripts/python[.exe])와 POSIX(.venv/bin/python) 양쪽 모두 매칭
_VENV_PATTERN = re.compile(
    r'\.?[\\/]?\.venv[\\/](Scripts|bin)[\\/]python[3]?(\.exe)?',
    re.IGNORECASE,
)
_ACTIVATE_PATTERN = re.compile(
    r'(source\s+)?\.?[\\/]?\.venv[\\/](Scripts|bin)[\\/]activate(\.(bat|ps1))?\s*(&&|;)\s*',
    re.IGNORECASE,
)

@tool
def run_command(command: str) -> str:
    """셸 명령어를 실행하고 결과를 반환합니다. 스크립트 실행 전용으로 사용하세요.
    파일을 만들거나 내용을 바꿀 때는 셸이 아니라 `write_file` 도구를 사용해야 합니다
    (윈도우 cmd의 명령줄 길이 제한 8191자, 따옴표 파싱 문제를 피하기 위해).

    Python 스크립트 실행 시 반드시 .venv의 Python을 사용하세요.
    예 (Windows): .venv/Scripts/python skills/sales_analytics/query.py "SELECT ..."
    예 (Linux/macOS): .venv/bin/python skills/sales_analytics/query.py "SELECT ..."

    Args:
        command: 실행할 명령어
    """
    try:
        # venv 경로와 활성화 명령을 현재 OS 경로로 정규화
        command = _ACTIVATE_PATTERN.sub('', command)
        command = _VENV_PATTERN.sub(_VENV_PYTHON.replace('\\', '/'), command)

        env = os.environ.copy()
        env["PYTHONIOENCODING"] = "utf-8"
        if IS_WINDOWS:
            # Windows cmd 한글 출력용 코드페이지 설정
            command = f"chcp 65001 >nul 2>&1 && {command}"

        result = subprocess.run(
            command, shell=True, capture_output=True, text=True,
            timeout=30, encoding="utf-8", errors="replace", env=env,
        )
        output = result.stdout
        if result.stderr:
            output += f"\n[STDERR] {result.stderr}"
        if result.returncode != 0:
            output += f"\n[EXIT CODE: {result.returncode}]"
        return output.strip() or "(no output)"
    except subprocess.TimeoutExpired:
        return "[TIMEOUT] 30초 초과"
    except Exception as e:
        return f"[ERROR] {e}"

print("도구 정의 완료: load_skill, run_command, write_file, read_file, list_files")
print(f"  OS: {platform.system()}")
print(f"  venv Python: {_VENV_PYTHON}")
print(f"  WORKSPACE: {WORKSPACE}")

도구 정의 완료: load_skill, run_command, write_file, read_file, list_files
  OS: Linux
  venv Python: /workspace/lab/.venv/bin/python
  WORKSPACE: /workspace/lab


In [6]:
from typing import Callable, Awaitable
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage

class SkillMiddleware(AgentMiddleware):
    """SKILL.md 기반 스킬 목록을 시스템 프롬프트에 주입하는 미들웨어.

    - tools = [load_skill]: 에이전트에 load_skill 도구를 자동 등록
    - wrap_model_call: 모델 호출 전 시스템 메시지에 스킬 목록 추가
    - awrap_model_call: stream_with_markdown / astream_events 등 비동기 호출 경로 지원
    """

    tools = [load_skill]

    def __init__(self, skills: dict[str, dict]):
        self.skills = skills
        lines = []
        for name, s in skills.items():
            skill_type = s["meta"].get("metadata", {}).get("type", "")
            type_tag = f" [{skill_type}]" if skill_type else ""
            lines.append(f"- `{name}`{type_tag}: {s['description']}")
        self.skills_prompt = "\n".join(lines)

    def _inject(self, request: ModelRequest) -> ModelRequest:
        """시스템 메시지에 스킬 목록을 덧붙여 새 request를 반환합니다."""
        addendum = (
            f"\n\n## Available Skills (SKILL.md 기반)\n\n{self.skills_prompt}\n\n"
            "특정 도메인의 작업을 수행하기 전에 `load_skill` 도구로 "
            "해당 스킬의 상세 정보(DB 스키마, 비즈니스 로직, 스크립트 사용법)를 먼저 로드하세요."
        )
        orig = request.system_message
        new_text = (orig.text + addendum) if orig else addendum
        return request.override(system_message=SystemMessage(content=new_text))

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        return handler(self._inject(request))

    async def awrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], Awaitable[ModelResponse]],
    ) -> ModelResponse:
        return await handler(self._inject(request))

print("SkillMiddleware 정의 완료")
print(f"  - tools: {[t.name for t in SkillMiddleware.tools]}")
print(f"  - 등록된 스킬: {list(SKILLS.keys())}")

SkillMiddleware 정의 완료
  - tools: ['load_skill']
  - 등록된 스킬: ['customer-report', 'data-viz', 'image-gen', 'inventory-management', 'sales-analytics']


---
## 3단계 Progressive Disclosure 흐름

스킬은 세 단계로 필요한 정보를 로드합니다.

1. 메타데이터: name과 description 목록만 시스템 프롬프트에 둡니다. SkillMiddleware가 주입합니다.
2. 본문: 작업이 필요하면 load_skill로 해당 SKILL.md 본문을 로드합니다.
3. 참고 파일: 본문이 가리키는 추가 문서는 read_file로 읽습니다. 
   load_skill이 스킬 폴더의 참고 파일 목록을 보여주므로, 에이전트가 필요한 것만 가져옵니다.

description은 모델이 스킬 로드를 판단할 때 쓰는 판단 신호입니다. 'When to Use'를 포함해야 합니다.

run_command는 스킬에 연결된 스크립트를 셸에서 직접 실행합니다. 신뢰할 수 없는 스킬은 샌드박스에서 격리해 실행해야 합니다.

---
## 4. Skill Agent 생성 및 테스트

에이전트에는 `run_command`만 직접 바인딩합니다.
`load_skill`은 `SkillMiddleware.tools`가 자동으로 등록하므로 따로 넣지 않습니다.

```
에이전트가 받는 도구: run_command + load_skill (middleware.tools 자동 등록)
시스템 프롬프트: 기본 지시 + 스킬 목록 (middleware.wrap_model_call 자동 주입)
```

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SKILLS = scan_skills(SKILLS_DIR)

skill_agent = create_agent(
    open_model,
    tools=[run_command, write_file, read_file, list_files],
    system_prompt=(
        "중간 과정에 대해 사용자에게 매번 설명하세요. "
    ),
    middleware=[SkillMiddleware(SKILLS)],
    checkpointer=InMemorySaver(),
)

print("Skill Agent 생성 완료")
print(f"  - 도구: run_command, write_file, read_file, list_files + load_skill (middleware 자동 등록)")
print(f"  - 스킬 {len(SKILLS)}개 감지")

Skill Agent 생성 완료
  - 도구: run_command, write_file, read_file, list_files + load_skill (middleware 자동 등록)
  - 스킬 5개 감지


### 4-1. CLI Tool 스킬 테스트: 매출 데이터 분석

스트리밍 출력에서 `load_skill` 호출과 쿼리 실행을 확인합니다.

기대 동작: `load_skill("sales_analytics")`로 DB 스키마를 확인한 뒤 `run_command`로 쿼리 실행

In [8]:
from stream_utils import stream_with_markdown

config = {"configurable": {"thread_id": "skill-test-1"}}

result = await stream_with_markdown(
    skill_agent,
    "지역별 매출 현황을 분석해줘. 가장 매출이 높은 지역과 낮은 지역을 비교해줘.",
    config=config,
)



지역별 매출 현황 분석을 진행하겠습니다. 먼저 판매 분석 스킬의 스키마와 비즈니스 로직을 확인하겠습니다.



🔧 Tool 호출: `load_skill` (인자: `{'skill_name': 'sales-analytics'}`)

✅ 결과: ## Tables

### customers
| 컬럼 | 타입 | 설명 |
|------|------|------|
| customer_id | INTEGER PK | 고객 ID |
| name | TEXT | 고객명 |
| email | TEXT | 이메일 |
| signup_date | DATE | 가입일 |
| status | TEXT | active



좋습니다. 이제 지역별 매출 현황을 분석하기 위해 SQL 쿼리를 실행하겠습니다. `completed` 상태인 주문들의 총 금액을 지역별로 집계합니다.



🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/sales_analytics/query.py "SELECT sales_region, COUNT(order_id) as order_count, SUM(total_amount) as total_revenue, AVG(total_amount) as avg_order_value FROM orders WHERE status = \'completed\' GROUP BY sales_region ORDER BY total_revenue DESC"'}`)

✅ 결과: sales_region | order_count | total_revenue | avg_order_value
-------------+-------------+---------------+----------------
west | 27 | 57311400.0 | 2122644.4444444445
east | 25 | 50193035.0 | 2007721.4


---


분석 결과가 나왔습니다. 지역별 매출 현황과 가장 높고 낮은 지역의 비교 결과를 정리해 드리겠습니다.

### 📊 지역별 매출 현황 요약

1.  **동부 (East)**: 50,193,035원 (25건)
2.  **남부 (South)**: 41,015,410원 (26건)
3.  **북부 (North)**: 31,494,440원 (18건)
4.  **서부 (West)**: 57,311,400원 (27건)

**출처:** `orders` 테이블 기준 (상태: completed)

---

### 🏆 가장 높은 지역 vs 가장 낮은 지역 비교

| 비교 항목 | 최고 지역 (**West**) | 최저 지역 (**North**) | 차액 / 비율 |
|-----------|----------------------|-----------------------|-------------|
| **총 매출** | 57,311,400원 | 31,494,440원 | **1.82배** 차이 |
| **주문 수** | 27건 | 18건 | 9건 차이 |
| **평균 주문가** | 약 212만 원 | 약 175만 원 | 약 37만 원 차이 |

#### 💡 분석 포인트
*   **매출량 격차**: 서부(West) 지역의 총 매출은 북부(North) 지역의 약 **1.8배**에 달합니다.
*   **주문 건수**: 서부 지역의 고객 활성도가 더 높아(27건 vs 18건), 단가 차이가 있지만 전체 매출에서 큰 우위를 점하고 있습니다.
*   **서부의 강점**: 서부는订单 수도 가장 많고 평균 단가도 가장 높아, 이 지역의 전략적 유지/보강이 필요해 보입니다. 반면 북부는 주문 수도 적고 매출 규모가 작아 마케팅 인센티브 등 추가 조사가 필요할 수 있습니다.

In [9]:
# 최종 응답 텍스트 
print(result)



지역별 매출 현황 분석을 진행하겠습니다. 먼저 판매 분석 스킬의 스키마와 비즈니스 로직을 확인하겠습니다.



좋습니다. 이제 지역별 매출 현황을 분석하기 위해 SQL 쿼리를 실행하겠습니다. `completed` 상태인 주문들의 총 금액을 지역별로 집계합니다.



분석 결과가 나왔습니다. 지역별 매출 현황과 가장 높고 낮은 지역의 비교 결과를 정리해 드리겠습니다.

### 📊 지역별 매출 현황 요약

1.  **동부 (East)**: 50,193,035원 (25건)
2.  **남부 (South)**: 41,015,410원 (26건)
3.  **북부 (North)**: 31,494,440원 (18건)
4.  **서부 (West)**: 57,311,400원 (27건)

**출처:** `orders` 테이블 기준 (상태: completed)

---

### 🏆 가장 높은 지역 vs 가장 낮은 지역 비교

| 비교 항목 | 최고 지역 (**West**) | 최저 지역 (**North**) | 차액 / 비율 |
|-----------|----------------------|-----------------------|-------------|
| **총 매출** | 57,311,400원 | 31,494,440원 | **1.82배** 차이 |
| **주문 수** | 27건 | 18건 | 9건 차이 |
| **평균 주문가** | 약 212만 원 | 약 175만 원 | 약 37만 원 차이 |

#### 💡 분석 포인트
*   **매출량 격차**: 서부(West) 지역의 총 매출은 북부(North) 지역의 약 **1.8배**에 달합니다.
*   **주문 건수**: 서부 지역의 고객 활성도가 더 높아(27건 vs 18건), 단가 차이가 있지만 전체 매출에서 큰 우위를 점하고 있습니다.
*   **서부의 강점**: 서부는订单 수도 가장 많고 평균 단가도 가장 높아, 이 지역의 전략적 유지/보강이 필요해 보입니다. 반면 북부는 주문 수도 적고 매출 규모가 작아 마케팅 인센티

### 4-2. Prompt Bundle 스킬 테스트: 보고서 생성

`customer_report` 스킬은 보고서 작성 가이드라인(Prompt Bundle)과 생성 스크립트(File Template)를 포함합니다.

기대 동작: `load_skill("customer_report")`로 가이드라인을 확인한 뒤 `generate.py`를 실행하고 결과 정리

In [14]:
config2 = {"configurable": {"thread_id": "skill-test-5"}}

result2 = await stream_with_markdown(
    skill_agent,
    "전체 매출과 재고 현황을 포함한 종합 보고서를 생성해줘. 보고서 가이드라인을 따라서 작성해. 이름은 report.md로 해.",
    config=config2,
)



종합 보고서를 생성하겠습니다. 먼저 보고서 가이드라인과 관련된 스킬들을 로드하고, 매출 및 재고 데이터를 조회하겠습니다.



🔧 Tool 호출: `load_skill` (인자: `{'skill_name': 'customer-report'}`)

🔧 Tool 호출: `load_skill` (인자: `{'skill_name': 'inventory-management'}`)

🔧 Tool 호출: `load_skill` (인자: `{'skill_name': 'sales-analytics'}`)

✅ 결과: ## Tables

### products
| 컬럼 | 타입 | 설명 |
|------|------|------|
| product_id | INTEGER PK | 상품 ID |
| product_name | TEXT | 상품명 |
| sku | TEXT | SKU 코드 |
| category | TEXT | 카테고리 |
| unit_cost | REAL 

✅ 결과: ## 보고서 구조

보고서는 반드시 아래 섹션 순서를 따릅니다:

1. **요약 (Executive Summary)** -- 핵심 지표 3~5개, 한 문단
2. **매출 분석** -- 기간별 매출 추이, 지역별/등급별 분포
3. **고객 분석** -- 활성 고객 수, CLV 상위 고객, 이탈 위험 고객
4. **재고 현황** -- 재주문 필요 상품, 창고별

✅ 결과: ## Tables

### customers
| 컬럼 | 타입 | 설명 |
|------|------|------|
| customer_id | INTEGER PK | 고객 ID |
| name | TEXT | 고객명 |
| email | TEXT | 이메일 |
| signup_date | DATE | 가입일 |
| status | TEXT | active



이제 모든 스킬 정보를 로드했습니다. 보고서에 필요한 데이터를 수집하겠습니다. 먼저 `generate.py` 스크립트를 사용하여 데이터 기반 보고서 초안을 생성합니다.



🔧 Tool 호출: `run_command` (인자: `{'command': 'skills/sales_analytics/setup.py && skills/inventory_management/setup.py && .venv/Scripts/python skills/customer_report/generate.py --type full'}`)

✅ 결과: [STDERR] /bin/sh: 1: skills/sales_analytics/setup.py: Permission denied

[EXIT CODE: 126]





🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/sales_analytics/setup.py && .venv/Scripts/python skills/inventory_management/setup.py && .venv/Scripts/python skills/customer_report/generate.py --type full'}`)

✅ 결과: [OK] /workspace/lab/skills/sales_analytics/sales.db 생성 완료 (고객 20명, 주문 200건)
[OK] /workspace/lab/skills/inventory_management/inventory.db 생성 완료 (상품 15개, 창고 3개, 이동 300건)
# 고객 분석 보고서

> 생성일: 2026-06-25





데이터가 잘 수집되었습니다. 이제 보고서를 완성하기 위해 추가 분석을 하겠습니다. 먼저 재주문이 필요한 상품과 창고별 재고를 조회하겠습니다.



🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/inventory_management/query.py "SELECT p.product_name, SUM(i.quantity_on_hand) as total_stock, p.reorder_point, (p.reorder_point - SUM(i.quantity_on_hand)) as shortage FROM inventory i JOIN products p ON i.product_id = p.product_id WHERE p.discontinued = 0 GROUP BY p.product_id HAVING total_stock <= p.reorder_point"'}`)

✅ 결과: product_name | total_stock | reorder_point | shortage
-------------+-------------+---------------+---------



재주문 필요 상품이 없습니다. 창고별 재고 분포를 확인하겠습니다.



🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/inventory_management/query.py "SELECT w.warehouse_name, p.product_name, i.quantity_on_hand, p.unit_cost * i.quantity_on_hand as stock_value FROM inventory i JOIN products p ON i.product_id = p.product_id JOIN warehouses w ON i.warehouse_id = w.warehouse_id WHERE p.discontinued = 0 ORDER BY w.warehouse_name, stock_value DESC"'}`)

✅ 결과: warehouse_name | product_name | quantity_on_hand | stock_value
---------------+--------------+------------------+------------
대전 물류센터 | 무선 이어폰 | 33 | 1485000.0
대전 물류센터 | 견과류 세트 | 71 | 1065000.0
대전 물류센



이제 고객 분석과 CLV 관련 데이터를 조회하겠습니다.



🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/sales_analytics/query.py "SELECT c.name, c.customer_tier, SUM(o.total_amount) as clv FROM customers c JOIN orders o ON c.customer_id = o.customer_id WHERE o.status = \'completed\' GROUP BY c.customer_id ORDER BY clv DESC LIMIT 5"'}`)

🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/sales_analytics/query.py "SELECT COUNT(*) FROM customers WHERE status=\'active\'"'}`)

🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/sales_analytics/query.py "SELECT strftime(\'%Y-%m\', order_date) as month, SUM(total_amount) as monthly_revenue FROM orders WHERE status = \'completed\' GROUP BY month ORDER BY month"'}`)


도구 실행 중...

In [15]:
# 최종 응답 텍스트
print(result2)



종합 보고서를 생성하겠습니다. 먼저 보고서 가이드라인과 관련된 스킬들을 로드하고, 매출 및 재고 데이터를 조회하겠습니다.



이제 모든 스킬 정보를 로드했습니다. 보고서에 필요한 데이터를 수집하겠습니다. 먼저 `generate.py` 스크립트를 사용하여 데이터 기반 보고서 초안을 생성합니다.





데이터가 잘 수집되었습니다. 이제 보고서를 완성하기 위해 추가 분석을 하겠습니다. 먼저 재주문이 필요한 상품과 창고별 재고를 조회하겠습니다.



재주문 필요 상품이 없습니다. 창고별 재고 분포를 확인하겠습니다.



이제 고객 분석과 CLV 관련 데이터를 조회하겠습니다.




### 4-3. 멀티턴 대화 테스트: 후속 분석 요청

같은 `thread_id`를 사용하여 앞선 대화의 맥락을 유지한 채 추가 분석을 요청합니다.

기대 동작: 앞선 대화에 포함된 스킬 본문을 참조해 추가 쿼리를 실행하고 세분화된 결과 반환

In [16]:
# 같은 thread_id로 후속 질문 실행
result3 = await stream_with_markdown(
    skill_agent,
    "아까 분석한 지역별 매출에서, 고객 등급(tier)별로 세분화해서 다시 분석해줘.",
    config=config,  # skill-test-1과 동일
)



네, 기존 분석 결과를 바탕으로 **고객 등급(Customer Tier)**을 기준으로 세분화하여 다시 분석해 드리겠습니다.

먼저 지역별(Tier별) 매핑된 SQL 쿼리를 실행해보겠습니다.



🔧 Tool 호출: `run_command` (인자: `{'command': '.venv/Scripts/python skills/sales_analytics/query.py "SELECT o.sales_region, c.customer_tier, COUNT(o.order_id) as order_count, SUM(o.total_amount) as total_revenue, AVG(o.total_amount) as avg_order_value FROM orders o JOIN customers c ON o.customer_id = c.customer_id WHERE o.status = \'completed\' GROUP BY o.sales_region, c.customer_tier ORDER BY o.sales_region, total_revenue DESC"'}`)

✅ 결과: sales_region | customer_tier | order_count | total_revenue | avg_order_value
-------------+---------------+-------------+---------------+----------------
east | bronze | 15 | 27688315.0 | 1845887.6666


---


고객 등급(Tier)을 고려한 세부 분석 결과입니다. `completed` 상태인 주문에 대해 고객 테이블(`customers`)과 조인하여 분석했습니다.

---

### 📊 1. 지역별 등급(Tier) 매출 현황

| Region | Top Tier (Highest Revenue) | Low Tier (Lowest Revenue) | Notable Tier Performance |
| :--- | :--- | :--- | :--- |
| **West** (서부) | 🥇 **Gold** (2,435만 원) | 🥉 Silver (1,021만 원) | Gold 등급에서 가장 높은 매출 기록 |
| **East** (동부) | 🥇 **Bronze** (2,768만 원) | 🥉 Gold (1,017만 원) | Bronze 등급이 2위(Silver)보다 2배 이상 상위 |
| **South** (남부) | 🥇 **Bronze** (2,389만 원) | 🥉 Gold (698만 원) | Bronze 등급 중심으로 판매량 유지 |
| **North** (북부) | 🥇 **Bronze** (1,728만 원) | 🥉 Gold (319만 원) | Gold 등급의 매출이 현저히 떨어짐 |

---

### 🔍 2. 주요 인사이트 및 교차 분석

#### A. **브론즈(Bronze) 등급: 'Quantity is King'**
*   동부(East), 남부(South), 북부(North)를 제외한 3개 지역에서 **총 매출 금액이 1위**를 차지했습니다.
*   이는 고등급 고객보다는 일반 고객(Bronze)의 거래 빈도가 훨씬 높음을 의미합니다. 즉, 서부를 제외하고는 **대중적인 마케팅**이 매출을 견인하고 있습니다.

#### B. **골드(Gold) 등급: 'West만의 특권'?**
*   흥미롭게도, **서부(West)** 지역에서는 Gold 등급이 Bronze(2,435만 vs 2,273만)와 비슷하거나 더 높은 매출을 올렸으며, **평균 주문가(Avg Order Value)도 243만 원으로 가장 높았습니다.**
*   반면 북부(North)와 남부(South)에서의 Gold 매출은 크게 부진합니다. (특히 북부 Gold는 319만 원에 그침)
*   **질문:** 북부와 남부의 골드 등급 고객을 위해 어떤 추가 혜택이나 리텐션(Retention) 프로그램이 필요할까요? 또는 해당 지역의 골드 고객층 자체가 부족할 수 있습니다.

#### C. **실버(Silver) 등급의 동부 활약**
*   동부(East)에서 실버(Silver) 등급의 **평균 주문가는 약 246만 원**으로 모든 조합 중 가장 높았습니다.
*   비록 총액은 브론즈에 밀렸지만, 고객 1인당 가치가 가장 높은 타겟 층이라는 점이 드러납니다.

---

### 💡 3. 전략적 제언

1.  **서부 지역 Gold 전략 강화**: 서부는 High-Value 고객(Gold)이 활발하게 움직이고 있습니다. 이 성공 사례를 Copy-Paste할 수 있는지가 핵심입니다.
2.  **북부/남부 Gold 리엔게이지먼트**: 두 지역에서 Gold 등급 customers가 상대적으로 적거나 소비가 위축되어 있습니다. 대상 정밀 분석(Diagnosis)이 필요합니다.
3.  **동부 Silver 업셀링(Upsell)**: 동부에서 Silver 등급의 평균 구매가가 매우 높으므로, 이들에게 Gold 업그레이드 인센티브를 제공하면 CLV(고객 생애 가치)가 크게 상승할 수 있을 것으로 보입니다.

In [17]:
# 최종 응답 텍스트
print(result3)



네, 기존 분석 결과를 바탕으로 **고객 등급(Customer Tier)**을 기준으로 세분화하여 다시 분석해 드리겠습니다.

먼저 지역별(Tier별) 매핑된 SQL 쿼리를 실행해보겠습니다.



고객 등급(Tier)을 고려한 세부 분석 결과입니다. `completed` 상태인 주문에 대해 고객 테이블(`customers`)과 조인하여 분석했습니다.

---

### 📊 1. 지역별 등급(Tier) 매출 현황

| Region | Top Tier (Highest Revenue) | Low Tier (Lowest Revenue) | Notable Tier Performance |
| :--- | :--- | :--- | :--- |
| **West** (서부) | 🥇 **Gold** (2,435만 원) | 🥉 Silver (1,021만 원) | Gold 등급에서 가장 높은 매출 기록 |
| **East** (동부) | 🥇 **Bronze** (2,768만 원) | 🥉 Gold (1,017만 원) | Bronze 등급이 2위(Silver)보다 2배 이상 상위 |
| **South** (남부) | 🥇 **Bronze** (2,389만 원) | 🥉 Gold (698만 원) | Bronze 등급 중심으로 판매량 유지 |
| **North** (북부) | 🥇 **Bronze** (1,728만 원) | 🥉 Gold (319만 원) | Gold 등급의 매출이 현저히 떨어짐 |

---

### 🔍 2. 주요 인사이트 및 교차 분석

#### A. **브론즈(Bronze) 등급: 'Quantity is King'**
*   동부(East), 남부(South), 북부(North)를 제외한 3개 지역에서 **총 매출 금액이 1위**를 차지했습니다.
*   이는 고등급 고객보다는 일반 고객(Bronze)의 거래 빈도가 훨씬 높음을 의미합니다. 즉, 서부를 제외하고는 **대중적인 마케팅**이 매출을 견인하고 있습니다.

#### B. **골드(Gold) 등급: 'West만의 특권

---
## [실습] 새로운 스킬 추가하기

Anthropic의 Skill Creator는 스킬 생성을 돕는 스킬입니다.

Smithery의 Skill Creator를 연결한 뒤 `Clock` 스킬을 생성하고 `load_skill`과 `run_command`로 실행합니다.

https://smithery.ai/skills/anthropics/skill-creator

예시: 현재 시간과 날짜를 파이썬 스크립트로 알려주는 `Clock` 스킬

In [ ]:
# TODO: skills/clock/SKILL.md와 clock.py를 만들고 load_skill/run_command로 실행
SKILLS = scan_skills(SKILLS_DIR)
skill_agent = create_agent(
    open_model,
    tools=[run_command, write_file, read_file, list_files],
    system_prompt="도구를 사용할 때마다 중간 과정을 간략히 설명하세요.",
    middleware=[SkillMiddleware(SKILLS)],
    checkpointer = InMemorySaver()
)

In [ ]:
config2 = {"configurable": {"thread_id": "skill-test-222"}}

result2 = await stream_with_markdown(
    skill_agent,
    "",  # TODO: 스킬 생성을 요청하세요.
    config=config2,
)

In [ ]:
# 최종 응답 텍스트
print(result2)

In [ ]:
# 생성한 스킬을 load_skill로 로드하고 run_command로 실행

---
## 정리

| 구성요소 | 역할 | Agent Skills 표준과의 대응 |
|----------|------|--------------------------|
| `SKILL.md` | YAML frontmatter + Markdown body로 스킬 정의 | `SKILL.md` 파일 형식 |
| `scan_skills()` | YAML frontmatter를 파싱하여 스킬 레지스트리 구성 | 스킬 메타데이터 탐색 |
| `load_skill` | 스킬 body + 스크립트 목록 로드 | `/skill-name` 호출 시 body 로드 |
| `run_command` | 스크립트 실행 | 에이전트의 run_command 도구로 실행 |
| `SkillMiddleware` | 시스템 프롬프트 주입 + 도구 자동 등록 | Progressive Disclosure |

### 정리 포인트

- SKILL.md 표준: `name`, `description`, `metadata` 등 구조화된 YAML 필드
- Progressive Disclosure: 스킬 요약 설명만 시스템 프롬프트에 주입하고, 필요한 스킬만 `load_skill`로 로드
- 유형: CLI Tool(스크립트 실행), Prompt Bundle(LLM 지침), File Template(양식 채우기)
- SkillMiddleware: `wrap_model_call`로 시스템 메시지에 스킬 목록 추가, `tools = [load_skill]`로 도구 자동 등록

## build_agent 익스텐션으로 스킬 배포

이번 파일에서는 Skill을 별도로 추가합니다.   

`EXTENSION_MODULES`에 적힌 모듈을 import해, 각 모듈의 `register()`가 돌려주는 미들웨어와 도구를 자동으로 추가합니다.   

skills.py가 SkillMiddleware와 스킬 도구를 `register()`로 전달합니다.

In [ ]:
%%writefile build_agent.py
"""build_agent.py

LLM + Tool + MCP + Middleware + SKill

MCP 도구를 await로 받아오므로 build_agent는 async 함수입니다.
"""

from __future__ import annotations

import os

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

from select_model import load_model
from tools import DEFAULT_TOOLS
from mcp_config import load_servers

DEFAULT_SYSTEM_PROMPT = (
    '당신은 유용한 AI 어시스턴트입니다. '
    '필요하면 도구를 사용하고, 도구를 실행하기 전 중간 과정을 간략히 설명하세요.'
)

# build_agent가 직접 연결하는 MCP 서버. mcp_servers.json의 stateless 서버를 읽는다.
# stateful 세션(Playwright 등)은 봇이 세션으로 열어 extra_tools로 넘긴다.
MCP_SERVERS = load_servers(stateful=False)

# 노트북 실습이 만든 배포용 익스텐션 모듈을 import해 미들웨어와 도구를 합친다.
# skills.py는 스킬 실습에서 만든다. 파일이 없거나 register()가 실패하면 그 모듈만
# 건너뛰고 나머지로 동작한다.
EXTENSION_MODULES = ['skills']


def load_extensions():
    """EXTENSION_MODULES의 각 모듈을 import해 register()가 돌려주는 미들웨어와 도구를 모읍니다.

    각 모듈은 register()에서 {'middleware': [...], 'tools': [...]} 형태를 돌려줍니다.
    import 실패나 register() 오류는 그 모듈만 건너뛰고 메시지를 출력합니다.
    """
    import importlib

    middleware, tools = [], []
    for name in EXTENSION_MODULES:
        try:
            mod = importlib.import_module(name)
        except Exception as e:
            print(f'[build_agent] 익스텐션 {name}을 건너뜁니다(로드 실패): {type(e).__name__}: {e}')
            continue
        register = getattr(mod, 'register', None)
        if register is None:
            print(f'[build_agent] 익스텐션 {name}에 register()가 없어 건너뜁니다')
            continue
        try:
            spec = register() or {}
        except Exception as e:
            print(f'[build_agent] 익스텐션 {name}을 건너뜁니다(register 오류): {type(e).__name__}: {e}')
            continue
        mw = list(spec.get('middleware', []))
        tl = list(spec.get('tools', []))
        middleware += mw
        tools += tl
        print(f'[build_agent] 익스텐션 {name} 적용: 미들웨어 {len(mw)}개, 도구 {len(tl)}개')
    return middleware, tools


async def _auto_elicitation(mcp_context, params, context):
    """Slack MCP의 post_message가 채널을 되물을 때 기본 채널로 자동 응답합니다.

    봇이나 노트북에는 사용자에게 입력 폼을 표시할 방법이 없으므로, SLACK_CHANNEL_ID
    환경변수의 채널로 답합니다.
    """
    from mcp.types import ElicitResult

    channel = os.environ.get('SLACK_CHANNEL_ID', 'C_DRYRUN')
    return ElicitResult(action='accept', content={'channel_id': channel})


async def load_mcp_tools(servers=None):
    """MCP 서버에서 도구를 받아옵니다. 연결에 실패하면 빈 목록을 돌려줍니다."""
    servers = MCP_SERVERS if servers is None else servers
    if not servers:
        return []
    try:
        from langchain_mcp_adapters.callbacks import Callbacks

        client = MultiServerMCPClient(
            servers, callbacks=Callbacks(on_elicitation=_auto_elicitation)
        )
        return await client.get_tools()
    except Exception as e:
        print(f'[build_agent] MCP 도구 로드 실패, 표준 도구만 사용합니다: {e}')
        return []


async def build_agent(
    model_name=None,
    platform='openai',
    model=None,
    tools=None,
    extra_tools=None,
    middleware=None,
    system_prompt=None,
    checkpointer=None,
    mcp_servers=None,
    **model_kwargs,
):
    """현재까지 구성된 에이전트를 만들어 반환합니다.

    EXTENSION_MODULES에 등록된 익스텐션이 있으면 그 미들웨어와 도구를 자동으로 더합니다.

    Args:
        model_name: 모델 이름. select_model.load_model로 전달됩니다.
        platform: 'openai' | 'vllm' | 'ollama'. load_model로 전달됩니다.
        model: 이미 만든 chat model. 주면 model_name/platform은 무시됩니다.
        tools: 표준 도구 대신 쓸 도구 목록. 생략 시 tools.py의 DEFAULT_TOOLS.
        extra_tools: 호출자가 직접 만든 도구를 추가로 붙입니다. 봇이 stateful
            세션(예: Playwright)에서 받은 도구를 넘길 때 씁니다.
        middleware: create_agent에 넘길 미들웨어 목록. 노트북에서 만든
            미들웨어를 넣으면 그 동작이 배포 에이전트에 적용됩니다.
        system_prompt: 시스템 프롬프트. 생략 시 기본값을 씁니다.
        checkpointer: 멀티턴 대화를 위한 체크포인터.
        mcp_servers: 연결할 MCP 서버 설정. 생략 시 MCP_SERVERS, {}면 MCP를 끕니다.
        model_kwargs: load_model로 전달되는 추가 인자 (reasoning_effort 등).
    """
    if model is None:
        model = load_model(model_name, platform=platform, **model_kwargs)
    base_tools = list(DEFAULT_TOOLS) if tools is None else list(tools)
    mcp_tools = await load_mcp_tools(mcp_servers)
    extra = list(extra_tools) if extra_tools else []
    ext_middleware, ext_tools = load_extensions()
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT
    return create_agent(
        model,
        tools=base_tools + mcp_tools + extra + ext_tools,
        middleware=(list(middleware) if middleware else []) + ext_middleware,
        system_prompt=system_prompt,
        checkpointer=checkpointer,
    )

In [ ]:
%%writefile skills.py
"""skills.py

배포 에이전트에 스킬 기능을 더하는 익스텐션입니다. skills/ 폴더의 SKILL.md를 스캔해
SkillMiddleware로 스킬 목록을 시스템 프롬프트에 주입하고, 스킬 스크립트를 실행하는
도구를 제공합니다. build_agent가 register()를 호출해 미들웨어와 도구를 받습니다.
"""

import os
import re
import sys
import platform
import subprocess
from pathlib import Path
from typing import Callable, Awaitable

import yaml
from langchain.tools import tool
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage

SKILLS_DIR = Path("skills")
IS_WINDOWS = platform.system() == "Windows"
WORKSPACE = Path.cwd().resolve()


def scan_skills(skills_dir: Path) -> dict[str, dict]:
    """SKILL.md의 frontmatter와 본문으로 스킬 레지스트리를 생성합니다."""
    skills = {}
    if not skills_dir.exists():
        return skills
    for d in sorted(skills_dir.iterdir()):
        if not d.is_dir():
            continue
        skill_file = d / "SKILL.md"
        if not skill_file.exists():
            continue
        content = skill_file.read_text(encoding="utf-8")
        if not content.startswith("---"):
            continue
        parts = content.split("---", 2)
        if len(parts) < 3:
            continue
        meta = yaml.safe_load(parts[1])
        body = parts[2].strip()
        name = meta.get("name", d.name)
        skills[name] = {
            "name": name,
            "description": meta.get("description", ""),
            "path": str(d),
            "folder": d.name,
            "meta": meta,
            "body": body,
        }
    return skills


def _safe_path(p: str) -> Path:
    """WORKSPACE 내부 경로로 정규화합니다."""
    target = (WORKSPACE / p).resolve()
    if not str(target).startswith(str(WORKSPACE)):
        raise ValueError(f"작업 디렉터리 밖 경로 접근 불가: {p}")
    return target


@tool
def load_skill(skill_name: str) -> str:
    """스킬의 상세 내용(SKILL.md body)을 로드합니다. 해당 도메인의 작업을 수행하기 전에 호출하세요.

    Args:
        skill_name: 로드할 스킬 이름 (YAML frontmatter의 name 필드)
    """
    skills = scan_skills(SKILLS_DIR)
    if skill_name not in skills:
        available = ", ".join(skills.keys())
        return f"스킬 '{skill_name}'을 찾을 수 없습니다. 사용 가능: {available}"

    skill = skills[skill_name]
    body = skill["body"]

    skill_dir = Path(skill["path"])
    scripts = [f.name for f in skill_dir.iterdir() if f.is_file() and f.suffix == ".py"]
    if scripts:
        body += "\n\n## 실행 가능한 스크립트\n"
        py_hint = ".venv\\Scripts\\python" if IS_WINDOWS else ".venv/bin/python"
        for s in scripts:
            body += f"- `{skill['folder']}/{s}`: `{py_hint} skills/{skill['folder']}/{s}`\n"

    ref_files = [
        f"skills/{skill['folder']}/{f.relative_to(skill_dir)}".replace("\\", "/")
        for f in sorted(skill_dir.rglob("*"))
        if f.is_file() and f.name != "SKILL.md" and f.suffix != ".py"
    ]
    if ref_files:
        body += "\n\n## 참고 파일\n필요할 때 read_file로 읽으세요. 작업에 필요한 파일만 로드합니다.\n"
        for rf in ref_files:
            body += f"- `{rf}`\n"

    return body


@tool
def write_file(path: str, content: str) -> str:
    """파일을 작성(덮어쓰기)합니다. 부모 디렉터리는 자동 생성됩니다.
    긴 내용의 SKILL.md, 스크립트 등을 만들 때 셸을 거치지 말고 이 도구를 사용하세요.

    Args:
        path: 작업 디렉터리 기준 상대 경로 (예: "skills/my_skill/SKILL.md")
        content: 파일에 쓸 전체 내용
    """
    try:
        target = _safe_path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return f"완료: {target.relative_to(WORKSPACE)} 작성 완료 ({len(content)}자)"
    except Exception as e:
        return f"[ERROR] {e}"


@tool
def read_file(path: str) -> str:
    """파일 내용을 읽어 반환합니다.

    Args:
        path: 작업 디렉터리 기준 상대 경로
    """
    try:
        return _safe_path(path).read_text(encoding="utf-8")
    except Exception as e:
        return f"[ERROR] {e}"


@tool
def list_files(path: str = ".") -> str:
    """디렉터리의 파일/하위 폴더 목록을 반환합니다.

    Args:
        path: 작업 디렉터리 기준 상대 경로 (기본값: 현재 디렉터리)
    """
    try:
        target = _safe_path(path)
        if not target.is_dir():
            return f"[ERROR] 디렉터리가 아님: {path}"
        entries = sorted(p.name + ("/" if p.is_dir() else "") for p in target.iterdir())
        return "\n".join(entries) if entries else "(empty)"
    except Exception as e:
        return f"[ERROR] {e}"


# OS에 맞는 venv Python 경로를 찾고, 없으면 현재 인터프리터를 쓴다.
if IS_WINDOWS:
    _candidate = os.path.join(os.getcwd(), ".venv", "Scripts", "python.exe")
else:
    _candidate = os.path.join(os.getcwd(), ".venv", "bin", "python")
_VENV_PYTHON = _candidate if os.path.isfile(_candidate) else sys.executable

# 윈도우(.venv/Scripts/python[.exe])와 POSIX(.venv/bin/python) 경로를 모두 매칭한다.
_VENV_PATTERN = re.compile(
    r'\.?[\\/]?\.venv[\\/](Scripts|bin)[\\/]python[3]?(\.exe)?',
    re.IGNORECASE,
)
_ACTIVATE_PATTERN = re.compile(
    r'(source\s+)?\.?[\\/]?\.venv[\\/](Scripts|bin)[\\/]activate(\.(bat|ps1))?\s*(&&|;)\s*',
    re.IGNORECASE,
)


@tool
def run_command(command: str) -> str:
    """셸 명령어를 실행하고 결과를 반환합니다. 스크립트 실행 전용으로 사용하세요.
    파일을 만들거나 내용을 바꿀 때는 셸이 아니라 `write_file` 도구를 사용해야 합니다
    (윈도우 cmd의 명령줄 길이 제한 8191자, 따옴표 파싱 문제를 피하기 위해).

    Python 스크립트 실행 시 반드시 .venv의 Python을 사용하세요.
    예 (Windows): .venv/Scripts/python skills/sales_analytics/query.py "SELECT ..."
    예 (Linux/macOS): .venv/bin/python skills/sales_analytics/query.py "SELECT ..."

    Args:
        command: 실행할 명령어
    """
    try:
        command = _ACTIVATE_PATTERN.sub('', command)
        command = _VENV_PATTERN.sub(_VENV_PYTHON.replace('\\', '/'), command)

        env = os.environ.copy()
        env["PYTHONIOENCODING"] = "utf-8"
        if IS_WINDOWS:
            command = f"chcp 65001 >nul 2>&1 && {command}"

        result = subprocess.run(
            command, shell=True, capture_output=True, text=True,
            timeout=30, encoding="utf-8", errors="replace", env=env,
        )
        output = result.stdout
        if result.stderr:
            output += f"\n[STDERR] {result.stderr}"
        if result.returncode != 0:
            output += f"\n[EXIT CODE: {result.returncode}]"
        return output.strip() or "(no output)"
    except subprocess.TimeoutExpired:
        return "[TIMEOUT] 30초 초과"
    except Exception as e:
        return f"[ERROR] {e}"


class SkillMiddleware(AgentMiddleware):
    """SKILL.md 기반 스킬 목록을 시스템 프롬프트에 주입하는 미들웨어.

    - tools = [load_skill]: 에이전트에 load_skill 도구를 자동 등록
    - wrap_model_call: 모델 호출 전 시스템 메시지에 스킬 목록 추가
    - awrap_model_call: 비동기 호출 경로 지원
    """

    tools = [load_skill]

    def __init__(self, skills: dict[str, dict]):
        self.skills = skills
        lines = []
        for name, s in skills.items():
            skill_type = s["meta"].get("metadata", {}).get("type", "")
            type_tag = f" [{skill_type}]" if skill_type else ""
            lines.append(f"- `{name}`{type_tag}: {s['description']}")
        self.skills_prompt = "\n".join(lines)

    def _inject(self, request: ModelRequest) -> ModelRequest:
        """시스템 메시지에 스킬 목록을 덧붙여 새 request를 반환합니다."""
        addendum = (
            f"\n\n## Available Skills (SKILL.md 기반)\n\n{self.skills_prompt}\n\n"
            "특정 도메인의 작업을 수행하기 전에 `load_skill` 도구로 "
            "해당 스킬의 상세 정보(DB 스키마, 비즈니스 로직, 스크립트 사용법)를 먼저 로드하세요."
        )
        orig = request.system_message
        new_text = (orig.text + addendum) if orig else addendum
        return request.override(system_message=SystemMessage(content=new_text))

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        return handler(self._inject(request))

    async def awrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], Awaitable[ModelResponse]],
    ) -> ModelResponse:
        return await handler(self._inject(request))


def register():
    """build_agent가 호출하는 진입점입니다. 스킬 미들웨어와 도구를 돌려줍니다.

    load_skill은 SkillMiddleware.tools로 자동 등록되므로 도구 목록에 넣지 않습니다.
    """
    return {
        "middleware": [SkillMiddleware(scan_skills(SKILLS_DIR))],
        "tools": [run_command, write_file, read_file, list_files],
    }

In [ ]:
# skills.py를 만든 뒤 build_agent를 다시 import합니다.
import importlib, build_agent as build_agent_module
importlib.reload(build_agent_module)
from build_agent import build_agent

# middleware나 tools를 직접 넘기지 않아도 build_agent가 skills.py를 자동으로 싣습니다.
agent = await build_agent(model=open_model, mcp_servers={})

result = await agent.ainvoke({
    'messages': [{'role': 'user', 'content': '사용 가능한 스킬을 알려줘.'}]
})
print(result['messages'][-1].text)